In [177]:
import pandas as pd
import os
from pathlib import Path
import re
from datetime import datetime
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import rasterio
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from torchmetrics.classification import MulticlassF1Score
from glob import glob
import torch
import random
import torch.nn.functional as F
import numpy as np
import torch.nn as nn

In [178]:
s1_pre_folder = './bs/sentinel1_pre/'
s1_post_folder = './bs/sentinel1_post/'
s2_pre_folder = './bs/sentinel2_pre/'
s2_post_folder = './bs/sentinel2_post/'
aux_folder = './bs/aux/'

ds_metadata = pd.read_csv("./bs/meta.csv")
ds_metadata_clean_s1 = pd.read_csv("./clean_s1.csv")

In [179]:
import rasterio
import numpy as np

idx = 21
s1_pre_prefix = "_Sentinel-1_pre"
s1_post_prefix = "_Sentinel-1_post"
s2_pre_prefix = "_Sentinel-2_pre"
s2_post_prefix = "_Sentinel-2_post"
aux_prefix = "_AUX"
path = aux_folder + ds_metadata_clean_s1.iloc[idx]["chip_id"] + aux_prefix + ".tif"

with rasterio.open(path) as src:
    s1_pre = src.read().astype(np.float32)  # shape: (C, H, W)
    print("shape:", s1_pre.shape, "dtype:", s1_pre.dtype)
    print("min/max:", s1_pre.min(), s1_pre.max())

shape: (3, 512, 512) dtype: float32
min/max: 4.0 95.0


In [180]:
import numpy as np
import rasterio

path = "bs/aux/BS_tr_000003_AUX.tif"  # подставь реальный путь

with rasterio.open(path) as src:
    aux = src.read().astype(np.float32)
    print("shape:", aux.shape, "dtype в файле:", src.dtypes, "nodata:", src.nodata)

for c in range(aux.shape[0]):
    ch = aux[c]
    finite = ch[np.isfinite(ch)]
    n_unique = len(np.unique(finite))
    print(f"\n--- канал {c} ---")
    print(f"min={finite.min():.2f} max={finite.max():.2f} mean={finite.mean():.2f}")
    print(f"n_unique={n_unique}")
    if n_unique <= 20:
        vals, counts = np.unique(finite, return_counts=True)
        print("уникальные значения и их частоты:", dict(zip(vals.tolist(), counts.tolist())))
    else:
        print("перцентили [1,25,50,75,99]:", np.percentile(finite, [1, 25, 50, 75, 99]))

shape: (3, 512, 512) dtype в файле: ('int16', 'int16', 'int16') nodata: None

--- канал 0 ---
min=-13.00 max=4.00 mean=-5.17
n_unique=18
уникальные значения и их частоты: {-13.0: 27532, -12.0: 523, -11.0: 2766, -10.0: 6764, -9.0: 14710, -8.0: 11386, -7.0: 13435, -6.0: 21161, -5.0: 25662, -4.0: 28859, -3.0: 36360, -2.0: 39721, -1.0: 21101, 0.0: 9513, 1.0: 1501, 2.0: 836, 3.0: 312, 4.0: 2}

--- канал 1 ---
min=1.00 max=1.00 mean=1.00
n_unique=1
уникальные значения и их частоты: {1.0: 262144}

--- канал 2 ---
min=20.00 max=90.00 mean=34.33
n_unique=6
уникальные значения и их частоты: {20.0: 104, 30.0: 227218, 40.0: 58, 60.0: 31517, 80.0: 469, 90.0: 2778}


In [203]:
import random
import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset

S2_SCALE = 10000.0
S1_SCALE = 100.0

S1_CLIP_RANGES = {"VV": (-25.0, 0.0), "VH": (-32.5, 0.0)}

S2_BAND_IDX = {
    "B2": 0, "B3": 1, "B4": 2, "B5": 3, "B6": 4,
    "B7": 5, "B8A": 6, "B11": 7, "B12": 8, "SCL": 9,
}

S2_DN_CLIP = (0.0, 10000.0)

SCL_INVALID = {3, 8, 9, 10, 11}

FOLDERS = {
    "s1_pre": ("./bs/sentinel1_pre/", "_Sentinel-1_pre"),
    "s1_post": ("./bs/sentinel1_post/", "_Sentinel-1_post"),
    "s2_pre": ("./bs/sentinel2_pre/", "_Sentinel-2_pre"),
    "s2_post": ("./bs/sentinel2_post/", "_Sentinel-2_post"),
    "aux": ("./bs/aux/", "_AUX"),
    "mask": ("./bs/masks/", "_MASK"),
}

def build_path(chip_id, kind):
    folder, suffix = FOLDERS[kind]
    return folder + chip_id + suffix + ".tif"

def load_tif(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)

def load_s2(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    arr = np.clip(arr, S2_DN_CLIP[0], S2_DN_CLIP[1])
    return arr

def normalize_s1_db_scaled(arr, band_indices={"VV": 0, "VH": 1}, scale=S1_SCALE):
    h, w = arr.shape[1], arr.shape[2]
    out = np.zeros((2, h, w), dtype=np.float32)
    for out_i, band in enumerate(("VV", "VH")):
        src_i = band_indices[band]
        band_db = arr[src_i] / scale
        lo, hi = S1_CLIP_RANGES[band]
        band_db = np.clip(band_db, lo, hi)
        out[out_i] = (band_db - lo) / (hi - lo)
    return out

def read_and_resample(band_path, match_meta):
    """Читает канал и, если нужно, изменяет его размер под геометрию match_meta."""
    with rasterio.open(band_path) as src:
        # Если размеры совпадают, читаем как есть
        if src.width == match_meta['width'] and src.height == match_meta['height']:
            return src.read(1).astype(np.float32)
        
        # Если размеры разные (например, B12 20м), делаем ресэмплинг до 10м
        data = src.read(
            1,
            out_shape=(match_meta['height'], match_meta['width']),
            resampling=Resampling.bilinear
        )
        return data.astype(np.float32)

def process_s1_pair(s1_pre_raw, s1_post_raw):
    return normalize_s1_db_scaled(s1_pre_raw), normalize_s1_db_scaled(s1_post_raw)


def build_valid_mask_scl(scl_pre, scl_post):
    invalid_pre = np.isin(scl_pre, list(SCL_INVALID))
    invalid_post = np.isin(scl_post, list(SCL_INVALID))
    return (~(invalid_pre | invalid_post)).astype(np.float32)


def process_aux_3ch(aux_arr, elevation_range):
    """aux_arr: (3, H, W) -> [dem, slope, land_cover_class]
    Порядок каналов подтверждён через rasterio descriptions: (dem, slope, landcover).
    Никакой маски валидности в AUX нет — валидность определяется только через SCL.
    """
    dem, slope, land_cover = aux_arr
    dem_norm = np.clip(
        (dem - elevation_range[0]) / (elevation_range[1] - elevation_range[0]),
        0, 1
    ).astype(np.float32)
    slope_norm = np.clip(slope / 90.0, 0, 1).astype(np.float32)  # градусы 0-90 -> [0,1]
    return dem_norm[None, ...], slope_norm[None, ...], land_cover.astype(np.int64)

def normalize_delta_s1(delta_db, clip_db=8.0):
    """
    delta_db: shape (H, W) или (1, H, W), уже в dB.
    Возвращает float32 в [-1, 1].
    """
    delta_db = np.clip(delta_db, -clip_db, clip_db)
    return (delta_db / clip_db).astype(np.float32)

IGNORE_INDEX = 255

class CosmohackDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def remap_mask(self, mask):
        return mask.astype(np.int64)  # BS-маска уже 0-3 согласно постановке кейса

    def apply_augmentations(self, x, mask):
        if random.random() > 0.5:
            x = np.flip(x, axis=2)
            mask = np.flip(mask, axis=1)
        if random.random() > 0.5:
            x = np.flip(x, axis=1)
            mask = np.flip(mask, axis=0)
        k = random.randint(0, 3)
        if k > 0:
            x = np.rot90(x, k, axes=(1, 2))
            mask = np.rot90(mask, k, axes=(0, 1))
        return np.ascontiguousarray(x), np.ascontiguousarray(mask)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        chip_id = row["chip_id"]
    
        s2_pre = load_s2(build_path(chip_id, "s2_pre")) / S2_SCALE
        s2_post = load_s2(build_path(chip_id, "s2_post")) / S2_SCALE

        s1_pre = load_tif(build_path(chip_id, "s1_pre")) / S1_SCALE
        s1_post = load_tif(build_path(chip_id, "s1_post")) / S1_SCALE

        vv_pre, vh_pre = s1_pre[0], s1_pre[1]
        vv_post, vh_post = s1_post[0], s1_post[1]
        
        dvv = normalize_delta_s1(vv_post - vv_pre, clip_db=8.0)[None, ...]
        dvh = normalize_delta_s1(vh_post - vh_pre, clip_db=8.0)[None, ...]
        
        idx_red = S2_BAND_IDX["B4"]
        idx_nir = S2_BAND_IDX["B8A"]
        idx_swir1 = S2_BAND_IDX["B11"]
        idx_swir2 = S2_BAND_IDX["B12"]
        idx_scl = S2_BAND_IDX["SCL"]
    
        pre_red = s2_pre[idx_red][None, ...]
        pre_nir = s2_pre[idx_nir][None, ...]
        pre_swir1 = s2_pre[idx_swir1][None, ...]
        pre_swir2 = s2_pre[idx_swir2][None, ...]
    
        post_red = s2_post[idx_red][None, ...]
        post_nir = s2_post[idx_nir][None, ...]
        post_swir1 = s2_post[idx_swir1][None, ...]
        post_swir2 = s2_post[idx_swir2][None, ...]
    
        scl_pre = (s2_pre[idx_scl] * S2_SCALE).round()
        scl_post = (s2_post[idx_scl] * S2_SCALE).round()
    
        pre_nbr1 = np.clip((pre_nir - pre_swir1) / (pre_nir + pre_swir1 + 1e-8), -1.0, 1.0,)
        post_nbr1 = np.clip((post_nir - post_swir1) / (post_nir + post_swir1 + 1e-8), -1.0, 1.0,)
    
        pre_nbr2 = np.clip((pre_nir - pre_swir2) / (pre_nir + pre_swir2 + 1e-8), -1.0, 1.0,)
        post_nbr2 = np.clip((post_nir - post_swir2) / (post_nir + post_swir2 + 1e-8), -1.0, 1.0)
    
        pre_ndvi = np.clip((pre_nir - pre_red) / (pre_nir + pre_red + 1e-8),-1.0, 1.0,)
        post_ndvi = np.clip((post_nir - post_red) / (post_nir + post_red + 1e-8),-1.0, 1.0,)
    
        dndvi = np.clip(pre_ndvi - post_ndvi, -2.0, 2.0)
        
        dnbr1 = np.clip(pre_nbr1 - post_nbr1, -2.0, 2.0)
        dnbr2 = np.clip(pre_nbr2 - post_nbr2, -2.0, 2.0)
    
        rdnbr1 = dnbr1 / (np.sqrt(np.abs(pre_nbr1)) + 1e-8)
        rdnbr1 = np.clip(rdnbr1, -6.0, 6.0) / 6.0

        rdnbr2 = dnbr2 / (np.sqrt(np.abs(pre_nbr2)) + 1e-8)
        rdnbr2 = np.clip(rdnbr2, -6.0, 6.0) / 6.0

        rdndvi = dndvi / (np.sqrt(np.abs(pre_ndvi)) + 1e-8)
        scl_valid_mask = build_valid_mask_scl(scl_pre, scl_post)[None, ...]
        
        rbr = dnbr2 / (pre_nbr2 + 1.001)
        rbr = np.clip(rbr, -2.0, 2.0) / 2.0
        rbr = np.nan_to_num(
            rbr,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(np.float32)
        # s1_pre_raw = load_tif(build_path(chip_id, "s1_pre"))
        # s1_post_raw = load_tif(build_path(chip_id, "s1_post"))
    
        # pre_norm_s1, post_norm_s1 = process_s1_pair(
            # s1_pre_raw,
            # s1_post_raw,
        # )
        # delta_s1 = (post_norm_s1 - pre_norm_s1).astype(np.float32)

    
        combined_valid_mask = scl_valid_mask
        x = np.concatenate([
            pre_nir,
            pre_swir1,
            pre_swir2,
    
            post_nir,
            post_swir1,
            post_swir2,
            
            rbr,
            rdnbr2,
            dvv,
            dvh,
            # rdndvi,
            
            combined_valid_mask,
        ], axis=0).astype(np.float32)
        
        mask_path = build_path(chip_id, "mask")
        mask = load_tif(mask_path).astype(np.int64)[0]
        mask = self.remap_mask(mask)
    
        valid_pixels = combined_valid_mask[0] > 0.5
    
        mask[~valid_pixels] = IGNORE_INDEX
    
        x[:, ~valid_pixels] = 0.0
    
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0,).astype(np.float32)
    
        if self.is_train:
            x, mask = self.apply_augmentations(x, mask)
    
        x_tensor = torch.from_numpy(x).float()
        mask_tensor = torch.from_numpy(mask).long()
    
        return x_tensor, mask_tensor

In [204]:
device = "cuda"

f1_metric = MulticlassF1Score(
    num_classes=4,
    average=None,
    ignore_index=255
).to(device)

## Архитектура Attention U-Net

In [205]:
class ConvBlock(torch.nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.conv = nn.Sequential(
                                  nn.Conv2d(ch_in, ch_out,
                                            kernel_size=3, stride=1,
                                            padding=1, bias=True),
                                  nn.BatchNorm2d(ch_out),
                                  nn.ReLU(inplace=True),
                                  nn.Conv2d(ch_out, ch_out,
                                            kernel_size=3, stride=1,
                                            padding=1, bias=True),
                                  nn.BatchNorm2d(ch_out),
                                  nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = self.conv(x)
        return x

In [206]:
class UpConvBlock(torch.nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.up = nn.Sequential(
                                nn.Upsample(scale_factor=2),
                                nn.Conv2d(ch_in, ch_out,
                                         kernel_size=3,stride=1,
                                         padding=1, bias=True),
                                nn.BatchNorm2d(ch_out),
                                nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = x = self.up(x)
        return x

In [207]:
class AttentionBlock(torch.nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        
        self.w_g = nn.Sequential(
                                nn.Conv2d(f_g, f_int,
                                         kernel_size=1, stride=1,
                                         padding=0, bias=True),
                                nn.BatchNorm2d(f_int)
        )
        
        self.w_x = nn.Sequential(
                                nn.Conv2d(f_l, f_int,
                                         kernel_size=1, stride=1,
                                         padding=0, bias=True),
                                nn.BatchNorm2d(f_int)
        )
        
        self.psi = nn.Sequential(
                                nn.Conv2d(f_int, 1,
                                         kernel_size=1, stride=1,
                                         padding=0,  bias=True),
                                nn.BatchNorm2d(1),
                                nn.Sigmoid(),
        )
        
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, g, x):
        g1 = self.w_g(g)
        x1 = self.w_x(x)
        psi = self.relu(g1+x1)
        psi = self.psi(psi)
        
        return psi*x

In [208]:
class AttentionUNet(torch.nn.Module):
    def __init__(self, n_classes=4, in_channel=11, out_channel=4):
        super().__init__() 
        
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv1 = ConvBlock(ch_in=in_channel, ch_out=64)
        self.conv2 = ConvBlock(ch_in=64, ch_out=128)
        self.conv3 = ConvBlock(ch_in=128, ch_out=256)
        self.conv4 = ConvBlock(ch_in=256, ch_out=512)
        self.conv5 = ConvBlock(ch_in=512, ch_out=1024)
        
        self.up5 = UpConvBlock(ch_in=1024, ch_out=512)
        self.att5 = AttentionBlock(f_g=512, f_l=512, f_int=256)
        self.upconv5 = ConvBlock(ch_in=1024, ch_out=512)
        
        self.up4 = UpConvBlock(ch_in=512, ch_out=256)
        self.att4 = AttentionBlock(f_g=256, f_l=256, f_int=128)
        self.upconv4 = ConvBlock(ch_in=512, ch_out=256)
        
        self.up3 = UpConvBlock(ch_in=256, ch_out=128)
        self.att3 = AttentionBlock(f_g=128, f_l=128, f_int=64)
        self.upconv3 = ConvBlock(ch_in=256, ch_out=128)
        
        self.up2 = UpConvBlock(ch_in=128, ch_out=64)
        self.att2 = AttentionBlock(f_g=64, f_l=64, f_int=32)
        self.upconv2 = ConvBlock(ch_in=128, ch_out=64)
        
        self.conv_1x1 = nn.Conv2d(64, out_channel,
                                  kernel_size=1, stride=1, padding=0)
        
    def forward(self, x):
        # encoder
        x1 = self.conv1(x)
        
        x2 = self.maxpool(x1)
        x2 = self.conv2(x2)
        
        x3 = self.maxpool(x2)
        x3 = self.conv3(x3)
        
        x4 = self.maxpool(x3)
        x4 = self.conv4(x4)
        
        x5 = self.maxpool(x4)
        x5 = self.conv5(x5)
        
        # decoder + concat
        d5 = self.up5(x5)
        x4 = self.att5(g=d5, x=x4)
        d5 = torch.concat((x4, d5), dim=1)
        d5 = self.upconv5(d5)
        
        d4 = self.up4(d5)
        x3 = self.att4(g=d4, x=x3)
        d4 = torch.concat((x3, d4), dim=1)
        d4 = self.upconv4(d4)
        
        d3 = self.up3(d4)
        x2 = self.att3(g=d3, x=x2)
        d3 = torch.concat((x2, d3), dim=1)
        d3 = self.upconv3(d3)
        
        d2 = self.up2(d3)
        x1 = self.att2(g=d2, x=x1)
        d2 = torch.concat((x1, d2), dim=1)
        d2 = self.upconv2(d2)
        
        d1 = self.conv_1x1(d2)
        
        return d1

## Функции потерь

In [209]:
from typing import Optional, Sequence
from torch import Tensor

class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-6, ignore_index=255):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        valid_mask = (target != self.ignore_index)

        safe_target = target.clone()
        safe_target[~valid_mask] = 0  # временно подменяем, чтобы one_hot не упал

        probs = F.softmax(logits, dim=1)
        target_oh = F.one_hot(safe_target, self.num_classes).permute(0, 3, 1, 2).float()

        valid_mask = valid_mask.unsqueeze(1).float()  # [B, 1, H, W]
        probs = probs * valid_mask
        target_oh = target_oh * valid_mask

        dims = (0, 2, 3)
        intersection = torch.sum(probs * target_oh, dims)
        cardinality = torch.sum(probs + target_oh, dims)
        dice = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1 - dice.mean()

class FocalLoss(nn.Module):
    def __init__(self,
                 alpha: Optional[Tensor] = None,
                 gamma: float = 0.,
                 reduction: str = 'mean',
                 ignore_index: int = -100):

        if reduction not in ('mean', 'sum', 'none'):
            raise ValueError(
                'Reduction must be one of: "mean", "sum", "none".')

        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.reduction = reduction

        self.nll_loss = nn.NLLLoss(
            weight=alpha, reduction='none', ignore_index=ignore_index)

    def __repr__(self):
        arg_keys = ['alpha', 'gamma', 'ignore_index', 'reduction']
        arg_vals = [self.__dict__[k] for k in arg_keys]
        arg_strs = [f'{k}={v!r}' for k, v in zip(arg_keys, arg_vals)]
        arg_str = ', '.join(arg_strs)
        return f'{type(self).__name__}({arg_str})'

    def forward(self, x: Tensor, y: Tensor) -> Tensor:
        if x.ndim > 2:
            # (N, C, d1, d2, ..., dK) --> (N * d1 * ... * dK, C)
            c = x.shape[1]
            x = x.permute(0, *range(2, x.ndim), 1).reshape(-1, c)
            # (N, d1, d2, ..., dK) --> (N * d1 * ... * dK,)
            y = y.view(-1)

        unignored_mask = y != self.ignore_index
        y = y[unignored_mask]
        if y.numel() == 0:
            return x.sum() * 0.0
        x = x[unignored_mask]

        log_p = F.log_softmax(x, dim=-1)
        ce = self.nll_loss(log_p, y)

        all_rows = torch.arange(len(x), device=x.device)
        log_pt = log_p[all_rows, y]

        pt = log_pt.exp()
        focal_term = (1 - pt)**self.gamma

        loss = focal_term * ce

        if self.reduction == 'mean':
            loss = loss.mean()
        elif self.reduction == 'sum':
            loss = loss.sum()

        return loss


def focal_loss(alpha: Optional[Sequence] = None,
               gamma: float = 2,
               reduction: str = 'mean',
               ignore_index: int = 255,
               device='cuda',
               dtype=torch.float32) -> FocalLoss:
    if alpha is not None:
        if not isinstance(alpha, Tensor):
            alpha = torch.tensor(alpha)
        alpha = alpha.to(device=device, dtype=dtype)

    fl = FocalLoss(
        alpha=alpha,
        gamma=gamma,
        reduction=reduction,
        ignore_index=ignore_index)
    return fl


class OHEM(nn.Module):
    def __init__(
        self,
        keep_ratio=0.7,
        class_weights=None,
        ignore_index=255,
        min_kept=1,
    ):
        super().__init__()

        if not 0.0 < keep_ratio <= 1.0:
            raise ValueError(
                "keep_ratio должен быть в диапазоне (0, 1]."
            )

        self.keep_ratio = keep_ratio
        self.class_weights = class_weights
        self.ignore_index = ignore_index
        self.min_kept = min_kept

    def forward(self, logits, target):
        pixel_loss = F.cross_entropy(
            logits,
            target,
            weight=self.class_weights,
            ignore_index=self.ignore_index,
            reduction="none",
        )

        valid_mask = target != self.ignore_index
        valid_loss = pixel_loss[valid_mask]

        if valid_loss.numel() == 0:
            return logits.sum() * 0.0

        n_valid = valid_loss.numel()

        n_keep = max(
            int(n_valid * self.keep_ratio),
            self.min_kept,
        )

        n_keep = min(
            n_keep,
            n_valid,
        )

        hard_losses, _ = torch.topk(
            valid_loss,
            k=n_keep,
            largest=True,
            sorted=False,
        )

        return hard_losses.mean()

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.2, beta=0.8, gamma_tversky=1.0, gamma_focal=2.0, ignore_index=255):
        """
        alpha: штраф за ложноположительные (FP).
        beta: штраф за ложноотрицательные (FN). 
              Если beta=0.7 (а alpha=0.3), модель сильнее штрафуется за пропуск редких классов (Recall приоритетнее).
        """
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma_tversky = gamma_tversky
        self.gamma_focal = gamma_focal
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        num_classes = logits.size(1)
        valid_mask = (targets != self.ignore_index)
        
        # Очищаем таргеты для совместимости с one_hot
        clean_targets = targets.clone()
        clean_targets[~valid_mask] = 0
        
        # 1. МНОГОКЛАССОВЫЙ FOCAL LOSS
        log_p = F.log_softmax(logits, dim=1)
        ce_loss = F.nll_loss(log_p, clean_targets, reduction='none')
        p = torch.exp(log_p)
        p_t = p.gather(1, clean_targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - p_t) ** self.gamma_focal
        focal_ce = (focal_weight * ce_loss)[valid_mask].mean()

        # 2. MULTI-CLASS TVERSKY LOSS
        probs = F.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(clean_targets, num_classes=num_classes).permute(0, 3, 1, 2).float()
        
        # Маскируем игнорируемые пиксели
        mask_expanded = valid_mask.unsqueeze(1).repeat(1, num_classes, 1, 1)
        probs = probs * mask_expanded
        targets_one_hot = targets_one_hot * mask_expanded
        
        # Суммируем по пространственным осям и батчу
        dims = (0, 2, 3)
        tp = torch.sum(probs * targets_one_hot, dim=dims)
        fp = torch.sum(probs * (1 - targets_one_hot), dim=dims)
        fn = torch.sum((1 - probs) * targets_one_hot, dim=dims)
        
        # Считаем индекс Тверского для каждого класса
        tversky_index = (tp + 1e-6) / (tp + self.alpha * fp + self.beta * fn + 1e-6)
        
        # Для классов выгорания [1:] можно применить чуть больший фокус, 
        # возведя (1 - tversky) в степень (gamma_tversky)
        tversky_loss_per_class = (1.0 - tversky_index) ** self.gamma_tversky
        
        # Усредняем лосс по всем классам
        tversky_loss = tversky_loss_per_class[1:].mean()

        # Итоговый гибридный лосс
        return focal_ce + tversky_loss

## Подсчёт весов

In [210]:
from torch.utils.data import WeightedRandomSampler

def calculate_sample_weights(df):   
    global_class_counts = np.zeros(4, dtype=np.int64)
    chip_stats = []
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        chip_id = row["chip_id"]
        mask_path = build_path(chip_id, "mask") 
        
        with rasterio.open(mask_path) as src:
            mask = src.read(1)
            
        counts = np.bincount(mask.ravel(), minlength=4)
        global_class_counts += counts
        chip_stats.append(counts)
    
    class_weights = 1.0 / (global_class_counts + 1)
    class_weights = class_weights / np.sum(class_weights)
    
    sample_weights = []
    for counts in chip_stats:
        weight = np.sum(counts * class_weights)
        sample_weights.append(weight)
        
    return sample_weights

def calculate_class_weights(df):
    counts = np.zeros(4, dtype=np.int64)

    for _, row in tqdm(df.iterrows(), total=len(df)):
        chip_id = row["chip_id"]

        mask = load_tif(build_path(chip_id, "mask"))[0].astype(np.int64)

        s2_pre = load_s2(build_path(chip_id, "s2_pre")) / S2_SCALE
        s2_post = load_s2(build_path(chip_id, "s2_post")) / S2_SCALE

        scl_pre = (s2_pre[S2_BAND_IDX["SCL"]] * S2_SCALE).round()
        scl_post = (s2_post[S2_BAND_IDX["SCL"]] * S2_SCALE).round()

        valid = build_valid_mask_scl(scl_pre, scl_post).astype(bool)

        valid_mask_values = mask[valid]
        counts += np.bincount(valid_mask_values, minlength=4)

    weights = counts.sum() / (len(counts) * np.maximum(counts, 1))
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32, device=device)

## Темпоральный сплит

In [211]:
df_first = pd.read_csv("./bs_temporal_split_train_2019_2023_val_2024.csv")
df_second = pd.read_csv("./bs_temporal_split_train_2019_2023_val_2024_clean_s1.csv")

split_map = df_first.set_index("chip_id")["split"]

df_second = df_second.copy()
df_second["split"] = df_second["chip_id"].map(split_map)

df_second.to_csv("bs_temporal_split_train_2019_2023_val_2024_clean_s1.csv")

In [212]:
import pandas as pd

SPLIT_PATH = "./bs_temporal_split_train_2019_2023_val_2024_clean_s1.csv"

saved_split = pd.read_csv(SPLIT_PATH)

print("Saved split:")
print(saved_split["split"].value_counts())

meta_with_dates = ds_metadata_clean_s1.copy()

meta_with_dates["date_post_dt"] = pd.to_datetime(meta_with_dates["date_post"], errors="coerce",)

meta_with_dates["year_post"] = meta_with_dates["date_post_dt"].dt.year

train_ids = saved_split.loc[saved_split["split"] == "train_temporal", "chip_id",]

val_ids = saved_split.loc[saved_split["split"] == "val_2024", "chip_id",]

train_temporal_df = meta_with_dates[meta_with_dates["chip_id"].isin(train_ids)].copy().reset_index(drop=True)

val_2024_df = meta_with_dates[meta_with_dates["chip_id"].isin(val_ids)].copy().reset_index(drop=True)

split_map = df_first.set_index("chip_id")["split"]

NUM_EPOCHS = 25

Saved split:
split
train_temporal    106
val_2024           70
Name: count, dtype: int64


## Датасеты / даталоудеры

In [213]:
train_ds = CosmohackDataset(train_temporal_df, is_train=True,)

val_ds = CosmohackDataset(val_2024_df, is_train=False,)

class_weights = calculate_class_weights(train_temporal_df)

train_sample_weights = calculate_sample_weights(train_temporal_df)

sampler = WeightedRandomSampler(weights=train_sample_weights, num_samples=len(train_sample_weights), replacement=True,)

train_loader = DataLoader(train_ds, batch_size=4, sampler=sampler, shuffle=False, num_workers=0, pin_memory=True)

val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0, pin_memory=True,)


print("Train dataset:", len(train_ds))
print("Val dataset:", len(val_ds))
print("Class weights:", class_weights.detach().cpu().numpy())

100%|██████████| 106/106 [00:00<00:00, 283.73it/s]

Train dataset: 106
Val dataset: 70
Class weights: [0.05342953 0.97931534 1.0263939  1.9408612 ]


In [217]:
class_weights[0] = class_weights[0] / 2
class_weights[1] = class_weights[1] * 3

## Обучение

In [218]:
import segmentation_models_pytorch as smp
NUM_CLASSES = 4
IGNORE_INDEX = 255
device = 'cuda'

best_bs_score = -float("inf")
best_miou_target = -float("inf")

amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

model = AttentionUNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
criterion = FocalTverskyLoss(alpha=0.2, beta=0.8, ignore_index=255)
# criterion = OHEM(keep_ratio=0.5, class_weights=class_weights)
#criterion = smp.losses.tversky(mode="multiclass", ignore_index=255, alpha=0.7, beta=0.3, gamma=1.5, class_weights=class_weights)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=NUM_EPOCHS)

accumulation_steps = 4  # Симулируем батч 4 * 4 = 16

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    
    for i, (images, masks) in enumerate(train_loader):
        images, masks = images.to(device, memory_format=torch.channels_last, non_blocking=True), masks.to(device, non_blocking=True)
        
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            outputs = model(images)
            loss = criterion(outputs, masks) / accumulation_steps
        
        if amp_dtype == torch.bfloat16:
            loss.backward()
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
        else:
            scaler.scale(loss).backward()
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        train_loss += loss.detach().item() * accumulation_steps
            
    model.eval()
    f1_metric.reset()
    val_loss = 0.0
    # IoU классов 0, 1, 2, 3.
    total_intersection = torch.zeros(NUM_CLASSES,device=device,)
    total_union = torch.zeros(NUM_CLASSES,device=device,)
    
    # Метрики бинарной задачи:
    # burn = target > 0
    burn_intersection = torch.tensor(0.0, device=device,)
    burn_union = torch.tensor(0.0, device=device,)
    burn_pred_pixels = torch.tensor(0.0, device=device,)
    burn_target_pixels = torch.tensor(0.0, device=device,)

    valid_pixels_total = torch.tensor(0.0, device=device,)
    
    val_loss = 0.0
    total_intersection = torch.zeros(4, device=device)
    total_union = torch.zeros(4, device=device)
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                outputs = model(images)
                val_loss += criterion(outputs, masks).item()
            preds = outputs.argmax(dim=1)
            f1_metric.update(outputs, masks)
            valid = masks != 255
            
            for c in range(NUM_CLASSES):
                pred_c = (preds == c) & valid
                target_c = (masks == c) & valid
                total_intersection[c] += (pred_c & target_c).sum()
                total_union[c] += (pred_c | target_c).sum()
                
            pred_burn = (preds > 0) & valid
            target_burn = (masks > 0) & valid
    
            burn_intersection += (pred_burn & target_burn).sum()
    
            burn_union += (pred_burn | target_burn).sum()
    
            burn_pred_pixels += pred_burn.sum()
            burn_target_pixels += target_burn.sum()

            valid_pixels_total += valid.sum()

    # IoU для классов 0, 1, 2, 3.
    iou_per_class = torch.where(total_union > 0, total_intersection / total_union, torch.full_like(total_union, float("nan")),).cpu().numpy()
    
    # Mean IoU severity-классов: 1, 2, 3.
    miou_severity = np.nanmean(iou_per_class[1:])
    
    # mIoU всех классов, включая фон.
    miou_all = np.nanmean(iou_per_class)
    
    # Бинарный IoU: burn против background.
    iou_burn = (burn_intersection / (burn_union + 1e-7)).item()
    
    # Дополнительная метрика: F1 именно для burned pixels.
    f1_burn = (2.0 * burn_intersection / (burn_pred_pixels + burn_target_pixels + 1e-7)).item()
    
    pred_burn_fraction = (burn_pred_pixels / valid_pixels_total).item()
    target_burn_fraction = (burn_target_pixels / valid_pixels_total).item()
    
    mean_ious = torch.where(total_union > 0, total_intersection / total_union, torch.full_like(total_union, float("nan"))).cpu().numpy()
    miou_target = np.nanmean(mean_ious[1:])
    f1_epoch = f1_metric.compute().cpu().numpy()
    val_loss /= len(val_loader)
    
    # Эта величина лежит в диапазоне [0, 0.65],
    # потому что 0.35 + 0.30 = 0.65.
    bs_score = (0.35 * iou_burn + 0.30 * miou_severity)
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    
    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS} | "
        f"LR: {current_lr:.7f}\n"
        f"Train Loss: {train_loss / len(train_loader):.4f} | "
        f"Val Loss: {val_loss:.4f}\n"
        f"IoU per class [0,1,2,3]: "
        f"{np.round(iou_per_class, 4)}\n"
        f"F1 per class [0,1,2,3]:  "
        f"{np.round(f1_epoch, 4)}\n"
        f"IoU burn: {iou_burn:.4f} | "
        f"F1 burn: {f1_burn:.4f}\n"
        f"mIoU all: {miou_all:.4f} | "
        f"mIoU severity [1-3]: {miou_severity:.4f}\n"
        f"Burn fraction: "
        f"pred={pred_burn_fraction:.4f}, "
        f"true={target_burn_fraction:.4f}\n"
        f"BSScore: {bs_score:.4f}"
    )

    if bs_score > best_bs_score:
        best_bs_score = bs_score
        best_miou_target = miou_severity
    
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "bs_score": best_bs_score,
                "iou_burn": iou_burn,
                "f1_burn": f1_burn,
                "miou_severity": miou_severity,
                "iou_per_class": iou_per_class,
                "class_weights": class_weights.detach().cpu(),
            },
            "best_temporal_focal-tversky_sampler_rbr.pt",
        )
    
        print(
            f"Checkpoint saved: "
            f"BSScore={best_bs_score:.4f} | "
            f"IoU_burn={iou_burn:.4f} | "
            f"mIoU_severity={miou_severity:.4f}"
        )

  0%|          | 0/25 [00:00<?, ?it/s]


Epoch 1/25 | LR: 0.0002988
Train Loss: 1.2920 | Val Loss: 2.2253
IoU per class [0,1,2,3]: [0.0239 0.0052 0.0302 0.0006]
F1 per class [0,1,2,3]:  [0.0468 0.0104 0.0586 0.0011]
IoU burn: 0.0922 | F1 burn: 0.1688
mIoU all: 0.0150 | mIoU severity [1-3]: 0.0120
Burn fraction: pred=0.9780, true=0.0904
BSScore: 0.0359


  4%|▍         | 1/25 [00:49<19:58, 49.95s/it]

Checkpoint saved: BSScore=0.0359 | IoU_burn=0.0922 | mIoU_severity=0.0120

Epoch 2/25 | LR: 0.0002953
Train Loss: 1.0610 | Val Loss: 1.0953
IoU per class [0,1,2,3]: [0.9142 0.0112 0.1123 0.2567]
F1 per class [0,1,2,3]:  [0.9552 0.0222 0.2019 0.4085]
IoU burn: 0.2745 | F1 burn: 0.4307
mIoU all: 0.3236 | mIoU severity [1-3]: 0.1267
Burn fraction: pred=0.0555, true=0.0904
BSScore: 0.1341


  8%|▊         | 2/25 [01:36<18:22, 47.95s/it]

...


Epoch 23/25 | LR: 0.0000047
Train Loss: 0.6432 | Val Loss: 0.7685
IoU per class [0,1,2,3]: [0.9376 0.2384 0.3886 0.6835]
F1 per class [0,1,2,3]:  [0.9678 0.3849 0.5597 0.812 ]
IoU burn: 0.5193 | F1 burn: 0.6836
mIoU all: 0.5620 | mIoU severity [1-3]: 0.4368
Burn fraction: pred=0.0944, true=0.0904
BSScore: 0.3128


 96%|█████████▌| 24/25 [17:21<00:40, 40.74s/it]


Epoch 24/25 | LR: 0.0000012
Train Loss: 0.6716 | Val Loss: 0.7557
IoU per class [0,1,2,3]: [0.9282 0.252  0.3956 0.6628]
F1 per class [0,1,2,3]:  [0.9628 0.4025 0.5669 0.7972]
IoU burn: 0.5129 | F1 burn: 0.6781
mIoU all: 0.5596 | mIoU severity [1-3]: 0.4368
Burn fraction: pred=0.1169, true=0.0904
BSScore: 0.3106


100%|██████████| 25/25 [18:09<00:00, 43.57s/it]


Epoch 25/25 | LR: 0.0000000
Train Loss: 0.6911 | Val Loss: 0.7697
IoU per class [0,1,2,3]: [0.9295 0.2541 0.3902 0.6794]
F1 per class [0,1,2,3]:  [0.9635 0.4053 0.5613 0.8091]
IoU burn: 0.5160 | F1 burn: 0.6808
mIoU all: 0.5633 | mIoU severity [1-3]: 0.4412
Burn fraction: pred=0.1150, true=0.0904
BSScore: 0.3130


In [225]:
def evaluate_model(model, loader, device, num_classes=4, ignore_index=255):
    model.eval()

    total_intersection = torch.zeros(num_classes, device=device)
    total_union = torch.zeros(num_classes, device=device)

    f1_metric = MulticlassF1Score(
        num_classes=num_classes,
        average=None,
        ignore_index=ignore_index,
    ).to(device)

    f1_metric.reset()

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(
                device,
                memory_format=torch.channels_last,
                non_blocking=True,
            )

            masks = masks.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=amp_dtype,
            ):
                outputs = model(images)

            preds = outputs.argmax(dim=1)

            f1_metric.update(outputs, masks)

            valid = masks != ignore_index

            for c in range(num_classes):
                pred_c = (preds == c) & valid
                target_c = (masks == c) & valid

                total_intersection[c] += (pred_c & target_c).sum()
                total_union[c] += (pred_c | target_c).sum()

    iou_per_class = torch.where(
        total_union > 0,
        total_intersection / total_union,
        torch.full_like(total_union, float("nan")),
    ).cpu().numpy()

    f1_per_class = f1_metric.compute().cpu().numpy()

    return {
        "iou_per_class": iou_per_class,
        "f1_per_class": f1_per_class,
        "miou_all": np.nanmean(iou_per_class),
        "miou_target": np.nanmean(iou_per_class[1:]),
    }

# state_dict = torch.load("best_temporal_comboloss_sampler_rdnbr_OHEM.pt", weights_only=False, map_location=device)
# trained_model = AttentionUNet(in_channel=9, out_channel=4).to(device)
# trained_model.load_state_dict(
#     state_dict["model_state_dict"],
#     strict=True,
# )
model.eval()
test_metrics = evaluate_model(
    model=model,
    loader=val_loader,
    device=device,
    num_classes=4,
    ignore_index=IGNORE_INDEX,
)

print("Test F1 per class:", np.round(test_metrics["f1_per_class"], 4))
print("Test IoU per class:", np.round(test_metrics["iou_per_class"], 4))
print("Test mIoU all:", round(test_metrics["miou_all"], 4))
print("Test target mIoU:", round(test_metrics["miou_target"], 4))

Test F1 per class: [0.9681 0.4084 0.5621 0.8075]
Test IoU per class: [0.9383 0.2566 0.3909 0.6772]
Test mIoU all: 0.5657
Test target mIoU: 0.4416


In [110]:
@torch.no_grad()
def get_confusion_matrix(
    model,
    loader,
    device,
    num_classes=4,
    ignore_index=255,
):
    model.eval()

    confusion = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.int64,
        device=device,
    )

    for images, masks in loader:
        images = images.to(
            device,
            memory_format=torch.channels_last,
            non_blocking=True,
        )

        masks = masks.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
        ):
            logits = model(images)

        preds = logits.argmax(dim=1)

        valid = masks != ignore_index

        y_true = masks[valid].long()
        y_pred = preds[valid].long()

        indices = y_true * num_classes + y_pred

        confusion += torch.bincount(
            indices,
            minlength=num_classes * num_classes,
        ).reshape(num_classes, num_classes)

    return confusion.cpu().numpy()

In [226]:
cm = get_confusion_matrix(
    model=model,
    loader=val_loader,
    device=device,
)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[10995116   217402   147380    44517]
 [  222107   175020    29285      678]
 [   74546    37516   229900    24837]
 [   17516       34    44672   277460]]


In [113]:
@torch.no_grad()
def evaluate_class1_bias(
    model,
    loader,
    device,
    class1_bias,
    num_classes=4,
    ignore_index=255,
):
    model.eval()

    confusion = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.int64,
        device=device,
    )

    for images, masks in loader:
        images = images.to(
            device,
            memory_format=torch.channels_last,
            non_blocking=True,
        )

        masks = masks.to(
            device,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
        ):
            logits = model(images)

        adjusted_logits = logits.clone()

        adjusted_logits[:, 1] += class1_bias

        preds = adjusted_logits.argmax(dim=1)

        valid = masks != ignore_index

        y_true = masks[valid].long()
        y_pred = preds[valid].long()

        flat_indices = (
            y_true * num_classes
            + y_pred
        )

        confusion += torch.bincount(
            flat_indices,
            minlength=num_classes * num_classes,
        ).reshape(
            num_classes,
            num_classes,
        )

    cm = confusion.float()

    true_count = cm.sum(dim=1)
    pred_count = cm.sum(dim=0)

    true_positive = torch.diag(cm)

    precision = true_positive / (
        pred_count + 1e-7
    )

    recall = true_positive / (
        true_count + 1e-7
    )

    f1 = (
        2.0 * precision * recall
        / ( precision + recall + 1e-7 )
    )

    union = (
        true_count
        + pred_count
        - true_positive
    )

    iou = true_positive / (
        union + 1e-7
    )

    burn_true_positive = cm[1:, 1:].sum()

    burn_true_count = cm[1:, :].sum()

    burn_pred_count = cm[:, 1:].sum()

    burn_union = (
        burn_true_count
        + burn_pred_count
        - burn_true_positive
    )

    iou_burn = (
        burn_true_positive
        / ( burn_union + 1e-7 )
    ).item()

    miou_severity = iou[1:].mean().item()

    bs_score = (
        0.35 * iou_burn
        + 0.30 * miou_severity
    )

    return {
        "class1_bias": class1_bias,
        "bs_score": bs_score,
        "iou_burn": iou_burn,
        "miou_severity": miou_severity,
        "iou_per_class": iou.cpu().numpy(),
        "precision_per_class": precision.cpu().numpy(),
        "recall_per_class": recall.cpu().numpy(),
        "f1_per_class": f1.cpu().numpy(),
        "iou_class1": iou[1].item(),
        "precision_class1": precision[1].item(),
        "recall_class1": recall[1].item(),
        "f1_class1": f1[1].item(),
        "confusion_matrix": confusion.cpu().numpy(),
    }

biases = np.arange(-0.10, 1.01, 0.05)

bias_results = []

checkpoint = torch.load(
    "best_temporal_focal-tversky_sampler_rbr.pt",
    map_location=device,
    weights_only=False,
)

trained_model = AttentionUNet(in_channel=8, out_channel=4).to(device)
trained_model.load_state_dict(checkpoint["model_state_dict"])
trained_model.eval()

for bias in biases:
    metrics = evaluate_class1_bias(
        model=trained_model,
        loader=val_loader,
        device=device,
        class1_bias=float(bias),
        num_classes=4,
        ignore_index=IGNORE_INDEX,
    )

    bias_results.append(metrics)

bias_table = pd.DataFrame([
    {
        "class1_bias": result["class1_bias"],
        "BSScore": result["bs_score"],
        "IoU_burn": result["iou_burn"],
        "mIoU_severity": result["miou_severity"],
        "IoU_class1": result["iou_class1"],
        "Precision_class1": result["precision_class1"],
        "Recall_class1": result["recall_class1"],
        "F1_class1": result["f1_class1"],
    }
    for result in bias_results
])

display(
    bias_table.round(4)
)

,class1_bias,BSScore,IoU_burn,mIoU_severity,IoU_class1,Precision_class1,Recall_class1,F1_class1
0,-0.10,0.3233,0.5389,0.4490,0.2116,0.4066,0.3060,0.3492
1,-0.05,0.3241,0.5384,0.4521,0.2202,0.3966,0.3312,0.3610
2,0.00,0.3246,0.5375,0.4551,0.2288,0.3862,0.3594,0.3723
3,0.05,0.3249,0.5360,0.4577,0.2364,0.3754,0.3897,0.3824
4,0.10,0.3249,0.5340,0.4599,0.2432,0.3648,0.4217,0.3912
5,0.15,0.3243,0.5311,0.4612,0.2473,0.3538,0.4512,0.3966
6,0.20,0.3227,0.5265,0.4615,0.2486,0.3403,0.4797,0.3982
7,0.25,0.3209,0.5215,0.4613,0.2487,0.3279,0.5076,0.3984
8,0.30,0.3184,0.5150,0.4603,0.2466,0.3145,0.5331,0.3956
9,0.35,0.3151,0.5073,0.4586,0.2427,0.3007,0.5571,0.3906


In [114]:
best_bs_row = bias_table.loc[
    bias_table["BSScore"].idxmax()
]

best_class1_row = bias_table.loc[
    bias_table["IoU_class1"].idxmax()
]

print("Best BSScore row:")
print(best_bs_row)

print("\nBest IoU class 1 row:")
print(best_class1_row)

Best BSScore row:
class1_bias         0.050000
BSScore             0.324907
IoU_burn            0.536011
mIoU_severity       0.457678
IoU_class1          0.236437
Precision_class1    0.375433
Recall_class1       0.389733
F1_class1           0.382449
Name: 3, dtype: float64

Best IoU class 1 row:
class1_bias         0.250000
BSScore             0.320921
IoU_burn            0.521477
mIoU_severity       0.461346
IoU_class1          0.248748
Precision_class1    0.327857
Recall_class1       0.507607
F1_class1           0.398396
Name: 7, dtype: float64


In [219]:
model = AttentionUNet().to(device)

checkpoint = torch.load(
    "best_temporal_focal-tversky_sampler_rbr.pt",
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

<All keys matched successfully>

In [220]:
df = pd.read_csv("./sbad/npy_tiles_512/tiles_df.csv")
df = df.reset_index(drop=True).copy()
df["chip_id"] = np.arange(len(df), dtype=np.int64)

In [221]:
df.to_csv("finetuning.csv")

In [223]:
finetune_ds = CosmohackDataset(
    df,
    is_train=True,
)

finetune_sample_weights = calculate_sample_weights(df)

finetune_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(
        finetune_sample_weights,
        dtype=torch.double,
    ),
    num_samples=len(finetune_sample_weights),
    replacement=True,
)

finetune_loader = DataLoader(
    finetune_ds,
    batch_size=4,
    sampler=finetune_sampler,
    num_workers=0,
    pin_memory=True,
)

  0%|          | 0/695 [00:00<?, ?it/s]


TypeError: can only concatenate str (not "int") to str

In [224]:
import os
import numpy as np

npy_folder = "./sbad/npy_tiles_512"

sample_id = "scene_0_y0_x0"

files = {
    "s2_pre": f"{sample_id}_pre.npy",
    "s2_post": f"{sample_id}_post.npy",
    "s1_pre": f"{sample_id}_s1pre.npy",
    "s1_post": f"{sample_id}_s1post.npy",
    "mask": f"{sample_id}_mask.npy",
}

for name, filename in files.items():
    path = os.path.join(npy_folder, filename)
    arr = np.load(path)

    print(
        f"{name:8s} | "
        f"shape={arr.shape} | "
        f"dtype={arr.dtype} | "
        f"min={np.nanmin(arr):.4f} | "
        f"max={np.nanmax(arr):.4f}"
    )

    if name == "mask":
        print("labels:", np.unique(arr))

s2_pre   | shape=(13, 512, 512) | dtype=float32 | min=0.0001 | max=1.0000
s2_post  | shape=(13, 512, 512) | dtype=float32 | min=0.0000 | max=1.0000
s1_pre   | shape=(4, 512, 512) | dtype=float32 | min=0.0000 | max=inf
s1_post  | shape=(4, 512, 512) | dtype=float32 | min=0.0000 | max=inf
mask     | shape=(1, 512, 512) | dtype=float32 | min=0.0000 | max=0.0000
labels: [0.]
